[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaskarjitsarmah/RL-Agents-Workshop-LLM/blob/main/notebooks/NB6_reward_hacking_and_safety.ipynb)

# NB6 - Reward Hacking, Robustness, and Safety Gates

Every result so far rests on one lucky fact: our reward is **verifiable**.
Execute the prediction, execute the gold, compare result sets. What we optimised
is what we wanted.

Most tasks are not like that. You will be asked to train an agent on something
where correctness cannot be checked mechanically, and you will write a
plausible-looking proxy instead.

This notebook is about what happens next.

> **The reward function is an attack surface, and the attacker is your own
> optimizer.**

It is not adversarial in any dramatic sense. Gradient descent is simply very good
at finding the cheapest thing that scores well - and if that is not what you
meant, it will find it anyway.

> **Restart the runtime before this notebook.** Colab does not free GPU memory between notebooks, and a leftover model from the previous one is the most common cause of an out-of-memory error halfway through a training run.
>
> *Runtime -> Restart session*, then run the setup cell below.

In [ ]:
# --- Setup. Safe to re-run. ---------------------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/RL-Agents-Workshop-LLM"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/bhaskarjitsarmah/RL-Agents-Workshop-LLM.git", "/content/RL-Agents-Workshop-LLM"], check=True)
    os.chdir("/content/RL-Agents-Workshop-LLM")
    # Install with uv, not pip: uv's resolver installs the pinned stack cleanly
    # on current Colab, where pip's resolver aborts the whole install (and then
    # the notebook runs on Colab's stock torch/transformers, which is the cause
    # of the "bitsandbytes>=..." / "torchvision::nms" / "unexpected dtype" errors).
    print("Installing the pinned stack (2-4 min the first time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system", "-q",
                    "-r", "requirements-colab.txt"], check=False)
else:
    # Run from the REPO ROOT in both environments, so every relative path in
    # every notebook ("data/...") means the same thing whether you are on Colab
    # (cwd = repo root) or opened the file locally from notebooks/.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

from llm_utils import build_db, preflight, capability, load_result, report_number
from llm_utils.plotting import use_house_style
import matplotlib.pyplot as plt

CAP = preflight()
use_house_style()
print("Database ready at:", build_db())
if not CAP["gpu"]:
    print()
    print("No GPU detected -> REPLAY MODE.")
    print("Training cells will load pre-baked runs; every chart still renders.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.")

if not CAP["wandb"]:
    os.environ.setdefault("WANDB_MODE", "offline")
    print()
    print("No WANDB_API_KEY -> W&B set to offline mode.")
    print("Training still runs and still logs; the curves land in ./wandb")
    print("instead of the cloud dashboard.")

In [ ]:
def baked(key, how):
    """Load a pre-baked run, or explain exactly how to produce it.

    Returns None when the artifact is missing. Callers must check -- we would
    rather show no chart than an invented one.
    """
    data = load_result(key)
    if data is None:
        print(f"[{key}] not baked yet.")
        print(f"  Produce it with:  {how}")
        print("  Then re-run this cell. (The pre-baked files ship with the repo;")
        print("   you only need this if you are rebuilding them yourself.)")
    return data


PREBAKED = not CAP["gpu"]   # charts get a watermark when we are replaying

## 1. A proxy reward that would pass code review

Here is a reward for text-to-SQL written by someone with no gold answers. Read
it and try to object.

```
0.4   the query executes without error
0.3   it returns a plausible number of rows (1..50)
0.3   its identifiers overlap the words of the question
```

Every term is defensible. "It should run." "It should return something, but not
the whole table." "It should be about what was asked."

It never compares the result to anything, because there is nothing to compare
to - that is the situation we are simulating.

In [ ]:
from llm_utils import r_hackable_rowcount, r_exec_match
from llm_utils.gen_tasks import read_jsonl

tasks = read_jsonl("data/tasks_train_gen.jsonl")[:80]
f = lambda s: f"```sql\n{s}\n```"

t = tasks[0]
print("QUESTION:", t["question"])
print("\n  gold                :", f"proxy={r_hackable_rowcount(f(t['gold']), t['question']):.2f}",
      f" true={r_exec_match(t['gold'], t['gold']):.0f}")

degenerate = "SELECT * FROM orders LIMIT 5;"
print("  SELECT * ... LIMIT 5 :",
      f"proxy={r_hackable_rowcount(f(degenerate), t['question']):.2f}",
      f" true={r_exec_match(degenerate, t['gold']):.0f}")

### The exploit

Keep the degenerate query. Alias the question's own nouns as column names to
farm the overlap term. Nothing about the answer improves.

In [ ]:
def stuff(question):
    nouns = [w.strip("?.,'") for w in question.lower().split() if len(w) > 4][:3]
    if not nouns:
        return "SELECT * FROM orders LIMIT 5;"
    return "SELECT " + ", ".join(f"order_id AS {w}" for w in nouns) + \
           " FROM orders LIMIT 5;"

beats = sum(1 for t in tasks
            if r_hackable_rowcount(f(stuff(t["question"])), t["question"])
            >= r_hackable_rowcount(f(t["gold"]), t["question"]))
mean_hack = sum(r_hackable_rowcount(f(stuff(t["question"])), t["question"])
                for t in tasks) / len(tasks)
mean_gold = sum(r_hackable_rowcount(f(t["gold"]), t["question"])
                for t in tasks) / len(tasks)
true_acc = sum(r_exec_match(stuff(t["question"]), t["gold"]) for t in tasks) / len(tasks)

print(f"proxy reward   gold {mean_gold:.3f}   keyword-stuffed junk {mean_hack:.3f}")
print(f"the junk matches or beats the gold on {beats}/{len(tasks)} tasks")
print(f"TRUE accuracy of the junk: {true_acc:.3f}")
print("\nA policy optimising this proxy has every incentive to become the junk.")

## 2. The scissors chart

Now train against that proxy for 50 steps, logging **both** the proxy reward we
optimise and the true validation accuracy we actually care about.

In [ ]:
hacked = baked("nb6_hacked_history",
                  "python scripts/bake_all.py --stage hacked")
if hacked:
    from llm_utils.plotting import scissors
    scissors(hacked, proxy_key="proxy_reward", truth_key="val_accuracy",
             prebaked=PREBAKED)
    plt.show()

If you saw only the left axis - which is exactly what your training dashboard
shows you by default - this run looks like a triumph.

**Reward is what you optimised. Accuracy is what you wanted. When they part
company, believe the accuracy.**

## 3. The gallery of degenerate winners

What did the hacked policy actually learn to say?

In [ ]:
from llm_utils import detect_reward_hacks

hacked_preds = baked("nb6_hacked_predictions",
                  "python scripts/bake_all.py --stage hacked")
if hacked_preds:
    rep = detect_reward_hacks(hacked_preds)
    print(f"suspicious: {rep['suspicious']}")
    print(f"distinct queries across {rep['n']} predictions: {rep['distinct_sql']}")
    print(f"most common ({rep['most_common_frac']:.0%} of all outputs):")
    print(f"  {rep['most_common_sql']}")
    print("\nflags:")
    for k, v in rep["flags"].items():
        print(f"  {k:<22} {v}")

**`answer_collapse` is the loudest single signal.** When a large share of outputs
are the *same* query, the policy has stopped reading the question - it found one
thing that scores well and settled there.

You can detect that without any gold answers at all, which makes it the cheapest
alarm to install on a run whose reward you do not fully trust.

## 4. Five mitigations, applied and measured

| # | mitigation | what it buys |
|---|---|---|
| 1 | **verifiable reward** | the real fix, when available |
| 2 | **KL anchor (`beta`)** | bounds how far the policy can drift from sense |
| 3 | **held-out validation gate** | refuses a checkpoint whose *true* accuracy fell |
| 4 | **output filters** | non-SELECT rejected; degenerate shapes flagged |
| 5 | **human-in-the-loop** | a person reads N trajectories and we measure agreement |

Number 3 is the direct descendant of repo 1's validation gate, and it is the one
that generalises: it does not care *why* the reward was wrong.

In [ ]:
from llm_utils import composite_reward, reward_bounds

print("mitigation 1 -- our real reward is separated:")
print("  ", reward_bounds())

print("\nmitigation 4 -- output filters:")
for sql in ("DROP TABLE customers;", "SELECT 1;", "SELECT * FROM orders LIMIT 5;"):
    r, parts = composite_reward(f"```sql\n{sql}\n```",
                                "SELECT name FROM customers WHERE city='Mumbai';")
    print(f"  {sql:<36} reward {r:5.2f}  unsafe={parts.get('unsafe', 0):g}")

In [ ]:
def validation_gate(history, patience=3, min_delta=0.0):
    # Promote only the checkpoint with the best TRUE validation accuracy.
    # Note what it does NOT look at: the training reward. That is the point --
    # a gate that trusted the reward would have promoted every step of the
    # hacked run above.
    best, best_step, since = -1.0, None, 0
    for h in history:
        acc = h.get("val_accuracy")
        if acc is None:
            continue
        if acc > best + min_delta:
            best, best_step, since = acc, h["step"], 0
        else:
            since += 1
            if since >= patience:
                return {"promote_step": best_step, "best_val": best,
                        "stopped_at": h["step"], "reason": "no val improvement"}
    return {"promote_step": best_step, "best_val": best,
            "stopped_at": history[-1]["step"] if history else None,
            "reason": "ran to completion"}

if hacked:
    g = validation_gate(hacked)
    print("gate on the HACKED run:", g)
    print(f"  -> would have stopped at step {g['stopped_at']} and promoted "
          f"step {g['promote_step']}")
    print("  The proxy reward kept climbing the whole time. The gate did not care.")

## 5. Human-in-the-loop

Automatic rewards are cheap and wrong in ways you cannot see from inside them.
Read a handful of trajectories yourself and measure how often you agree.

Disagreement is not noise - it is a specification bug you have not written down
yet.

In [ ]:
hitl = baked("nb6_human_labels",
                  "python scripts/bake_all.py --stage robustness")
if hitl:
    tp = sum(1 for r in hitl if r["human"] and r["auto"])
    tn = sum(1 for r in hitl if not r["human"] and not r["auto"])
    fp = sum(1 for r in hitl if not r["human"] and r["auto"])
    fn = sum(1 for r in hitl if r["human"] and not r["auto"])
    n = len(hitl)
    print(f"                auto=good  auto=bad")
    print(f"  human=good  {tp:>9}  {fn:>9}")
    print(f"  human=bad   {fp:>9}  {tn:>9}")
    print(f"\nagreement: {(tp + tn) / n:.0%} over {n} trajectories")
    print("Every off-diagonal cell is a place your reward and your intent differ.")

## 6. Robustness: does the gain survive contact with reality?

A policy tuned hard on one phrasing distribution can be brittle. We perturb the
16 test questions four ways and re-score every checkpoint:

- **paraphrase** - same question, different words
- **typo** - realistic keyboard slips
- **distractor** - an irrelevant clause bolted on
- **schema rename** - a column referred to by a synonym

The perturbations are generated once, hand-checked, and frozen in
`data/test_perturbed.json`, so these numbers are deterministic and runnable
offline.

In [ ]:
rob = baked("nb6_robustness",
                  "python scripts/bake_all.py --stage robustness")
if rob:
    import numpy as np
    kinds = ["clean", "paraphrase", "typo", "distractor", "rename"]
    models = list(rob)
    x = np.arange(len(kinds)); w = 0.8 / max(len(models), 1)
    plt.figure(figsize=(10, 4))
    for i, m in enumerate(models):
        plt.bar(x + i * w, [rob[m].get(k, 0) for k in kinds], w, label=m)
    plt.xticks(x + 0.4 - w / 2, kinds); plt.ylabel("accuracy")
    plt.axhline(0.75, ls="--", color="#8C8C8C")
    plt.title("Robustness across perturbations"); plt.legend(); plt.tight_layout()
    plt.show()

## Takeaways

1. **A reward that cannot be checked will be gamed**, and the gaming looks like success on your training dashboard.
2. The scissors chart is the diagnostic: **log the true metric alongside the optimised one**, always, even when you are confident.
3. `answer_collapse` - a large share of identical outputs - detects hacking with no gold answers at all. Cheapest alarm you can install.
4. **Gate promotion on held-out validation accuracy, not training reward.** It does not need to know why the reward was wrong.
5. This is repo 1's lesson with a different parameter vector: **an ungated optimizer finds the cheapest thing that scores well, whether the parameter is text or floats.**

### The gap this leaves (-> NB7)

We have an adapter that is honestly better, and we know it is honestly better
because we checked it the hard way.

Nobody can use a `.safetensors` file sitting in a Colab VM. Next: merge it,
serve it, and find out what it actually costs to run - because accuracy is one
axis and the other two decide whether this ships.

### Exercise

1. Write your own proxy reward that looks *more* reasonable than
   `r_hackable_rowcount` - add a column-count check, say. Then try to break it.
   How long did that take?
2. The validation gate uses `patience=3`. On the hacked run, what is the largest
   patience that still stops before the true accuracy has meaningfully fallen?
3. Add a `no_where_no_join` penalty to the reward and re-run the hacked training.
   Does the policy find a different exploit? (It usually does. That is the
   lesson: patching exploits is not the same as fixing the objective.)

In [ ]:
# --- Cost / throughput meter -------------------------------------------
from llm_utils import METER, flush
print(METER)          # OpenAI spend (0 unless you ran the comparison rows)
try:
    import wandb; wandb.finish()
except Exception:
    pass
flush()